In [1]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q statsmodels


!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('/kaggle/working/ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

# Gemma 3
#!ollama pull gemma3:1b
!ollama pull gemma3:4b
#!ollama pull gemma3:12b

# Qwen 2.5
#!ollama pull qwen2.5:1.5b
!ollama pull qwen2.5:7b
#!ollama pull qwen2.5:14b

# Llama 3.2
#!ollama pull llama3.2:1b
!ollama pull llama3.2:3b

# Phi-4
!ollama pull phi4-mini


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 708.7 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 68.9 MB/s eta 0:00:00:00:010:01
Selecting previously unselected package zstd.
(Reading database ... 129802 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%##                                  55.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Service Ollama demarre
]11;?\pulling

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter
from statsmodels.stats.contingency_tables import cochrans_q
from scipy.stats import chi2
import numpy as np
import os, datetime
from urllib.parse import urlparse

In [3]:
!nvidia-smi

Thu Oct  8 02:31:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [6]:
chemin_in = "/kaggle/input/datasets/imenec/echantillon-data/"
chemin_out = "/kaggle/working/"

articles = lire_csv(chemin_in + "echantillon_150.csv")

print(f"Echantillon  : {len(articles)} articles")


Echantillon  : 150 articles


In [8]:
def chercher_preuves(requete):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points

    #requete = titre if titre else texte[:100]
    try:
        resultats = DDGS().text(requete, max_results=3)
    except Exception as e:
        print(f"  Erreur de recherche : {e}")
        return "(aucune preuve trouvee)"

    if not resultats:
        return "(aucune preuve trouvee)"

    morceaux = []
    for r in resultats:
        morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
    return "\n".join(morceaux)


In [7]:
def chercher_preuves_2(requete, max_tentatives=3):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points
    # difference avec l'autre, si on échoue a trouver des preuves, on essaye jusqu'a 3 tentatives


    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout augmente a 20s (au lieu de 5s par defaut)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(aucune preuve trouvee)", []

            morceaux = []
            domaines = []
            
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
                domaines.append(urlparse(r.get("href", "")).netloc)
            return "\n".join(morceaux), domaines

        except Exception as e:
            print(f"  [!] Tentative {tentative}/{max_tentatives} echouee : {e}")
            if tentative < max_tentatives:
                # attend de plus en plus longtemps a chaque tentative (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # si toutes les tentatives ont echoue
    return "(aucune preuve trouvee apres plusieurs tentatives)", []


In [9]:
def construire_prompt(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    if source != "":
        prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    else:
        prompt = prompt + "En te basant sur ce que tu connais déja, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower().strip()

    contient_fake = bool(re.search(r"\bfake\b", texte))
    contient_real = bool(re.search(r"\breal\b", texte))

    # uniquemnt fake
    if contient_fake and not contient_real:
        return "fake"

    # uniquemnt real
    if contient_real and not contient_fake:
        return "real"

    # sinon
    return "indetermine"

In [10]:
def demander_verdict(titre, texte, source, preuves, avec_rag):

    prompt = construire_prompt(titre, texte, source, preuves, avec_rag)

    #reponse = ollama.chat(model=m, messages=[{"role": "user", "content": prompt}])

    reponse = ollama.chat(
        model=m,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu)


In [11]:
FICHIER_PREUVES = chemin_out + "preuves_150.csv"

def sauver_preuve(id_article, requete, preuves, domaines):
    nouveau = not os.path.exists(FICHIER_PREUVES)
    with open(FICHIER_PREUVES, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nouveau:
            w.writerow(["id", "requete", "Preuves", "domaine", "date_recherche"])
        w.writerow([id_article, requete, preuves, " | ".join(domaines), datetime.date.today().isoformat()])

In [12]:
def nettoyer_requete(titre):
    # enlève les noms de sites de fact-checking qui revelent la source
    titre = re.sub(r"gossip\s*cop|politi\s*fact|snopes", "", titre, flags=re.IGNORECASE)
    titre = re.sub(r"\s+", " ", titre)      #espaces en double
    return titre.strip(" -|:–—")              # tirets ou deux-points restes aux bords

In [14]:
def nom_media(domaine):
    domaine = domaine.lower()
    domaine = domaine.replace("www.", "")
    return domaine.split(".")[0]

In [15]:
def tester_contrefactuel(article,sources_connues_credibles,sources_connues_peu_credibles,
                         sources_peu_connues_credibles, sources_peu_connues_peu_credibles, avec_rag, creer_preuves):

    titre = article.get("title","")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # -------------------------
    # RAG ou non-RAG
    # -------------------------
    if avec_rag:
        if creer_preuves:
            requete = titre if titre else texte[:100]
            preuves, domaines = chercher_preuves_2(requete)

            # Sauvegarde en memoire
            article["preuves"] = preuves

            # Sauvegarde dans le fichier CSV
            sauver_preuve(article["id"],requete,preuves,domaines)
            print(f"RAG CRÉÉ pour : {article.get('id', '')}")

        else:
            # reutilisaton des memes preuves
            preuves = article["preuves"]
            print(f"RAG RÉUTILISÉ pour : {article.get('id', '')}")

    else:
        preuves = None


    #Source originale
    verdict_original = demander_verdict(titre, texte, vraie_source, preuves, avec_rag)

    #Connue et crédible
    verdicts_connue_credible = []
    for source in sources_connues_credibles:
        #on demande un verdict pour nos 3 sources connues et creédiles
        verdict = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts_connue_credible.append(verdict)

    
    # Connue et peu crédible
    verdicts_connue_peu_credible = []
    for source in sources_connues_peu_credibles:
        #on demande un verdict pour nos 3 sources connues et peu creédiles
        verdict = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts_connue_peu_credible.append(verdict)

    
    #Peu connue et crédible
    verdicts_peu_connue_credible = []
    for source in sources_peu_connues_credibles:
        #on demande un verdict pour nos 3 sources peu connues mais creédiles
        verdict = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts_peu_connue_credible.append(verdict)

    
    # Peu connue et peu crédible
    verdicts_peu_connue_peu_credible = []
    for source in sources_peu_connues_peu_credibles:
        #on demande un verdict pour nos 3 sources peu connues et peu creédiles
        verdict = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts_peu_connue_peu_credible.append(verdict)

    
    # Sans source
    pas_source=""
    verdict_sans_source = demander_verdict(titre, texte, pas_source, preuves, avec_rag)


    return {
    "id": article.get("id", ""),
    "vraie_source": vraie_source,
    "vrai_label": vrai_label,

    "verdict_original": verdict_original,

    "source_connue_credible_1": sources_connues_credibles[0],
    "verdict_connue_credible_1": verdicts_connue_credible[0],
    "source_connue_credible_2": sources_connues_credibles[1],
    "verdict_connue_credible_2": verdicts_connue_credible[1],
    "source_connue_credible_3": sources_connues_credibles[2],
    "verdict_connue_credible_3": verdicts_connue_credible[2],

    "source_connue_peu_credible_1": sources_connues_peu_credibles[0],
    "verdict_connue_peu_credible_1": verdicts_connue_peu_credible[0],
    "source_connue_peu_credible_2": sources_connues_peu_credibles[1],
    "verdict_connue_peu_credible_2": verdicts_connue_peu_credible[1],
    "source_connue_peu_credible_3": sources_connues_peu_credibles[2],
    "verdict_connue_peu_credible_3": verdicts_connue_peu_credible[2],

    "source_peu_connue_credible_1": sources_peu_connues_credibles[0],
    "verdict_peu_connue_credible_1": verdicts_peu_connue_credible[0],
    "source_peu_connue_credible_2": sources_peu_connues_credibles[1],
    "verdict_peu_connue_credible_2": verdicts_peu_connue_credible[1],
    "source_peu_connue_credible_3": sources_peu_connues_credibles[2],
    "verdict_peu_connue_credible_3": verdicts_peu_connue_credible[2],

    "source_peu_connue_peu_credible_1": sources_peu_connues_peu_credibles[0],
    "verdict_peu_connue_peu_credible_1": verdicts_peu_connue_peu_credible[0],
    "source_peu_connue_peu_credible_2": sources_peu_connues_peu_credibles[1],
    "verdict_peu_connue_peu_credible_2": verdicts_peu_connue_peu_credible[1],
    "source_peu_connue_peu_credible_3": sources_peu_connues_peu_credibles[2],
    "verdict_peu_connue_peu_credible_3": verdicts_peu_connue_peu_credible[2],

    "source_sans_source": pas_source,
    "verdict_sans_source": verdict_sans_source
}

In [16]:
def lancer_test_contrefactuel(echantillon, chemin_sortie, sources_connues_credibles, sources_connues_peu_credibles,
                              sources_peu_connues_credibles, sources_peu_connues_peu_credibles, avec_rag, creer_preuves):

    colonnes = [
        "id",
        "vraie_source",
        "vrai_label",
        "verdict_original",
    
        "source_connue_credible_1",
        "verdict_connue_credible_1",
        "source_connue_credible_2",
        "verdict_connue_credible_2",
        "source_connue_credible_3",
        "verdict_connue_credible_3",
    
        "source_connue_peu_credible_1",
        "verdict_connue_peu_credible_1",
        "source_connue_peu_credible_2",
        "verdict_connue_peu_credible_2",
        "source_connue_peu_credible_3",
        "verdict_connue_peu_credible_3",
    
        "source_peu_connue_credible_1",
        "verdict_peu_connue_credible_1",
        "source_peu_connue_credible_2",
        "verdict_peu_connue_credible_2",
        "source_peu_connue_credible_3",
        "verdict_peu_connue_credible_3",
    
        "source_peu_connue_peu_credible_1",
        "verdict_peu_connue_peu_credible_1",
        "source_peu_connue_peu_credible_2",
        "verdict_peu_connue_peu_credible_2",
        "source_peu_connue_peu_credible_3",
        "verdict_peu_connue_peu_credible_3",
    
        "source_sans_source",
        "verdict_sans_source"
    ]

    resultats = []

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(f, fieldnames=colonnes)
        writer.writeheader()

        for i, article in enumerate(echantillon):

            r = tester_contrefactuel( article, sources_connues_credibles, sources_connues_peu_credibles,
                                     sources_peu_connues_credibles, sources_peu_connues_peu_credibles, avec_rag, 
                                     creer_preuves)

            writer.writerow(r)
            resultats.append(r)

            print(f"[{i+1}/{len(echantillon)}] OK")

    return resultats

In [17]:

sources_connues_credibles = [
    "reuters.com",
    "cnn.com",
    "nytimes.com"
]

sources_connues_peu_credibles = [
    "dailymail.co.uk",
    "thesun.co.uk",
    "breitbart.com"
]

sources_peu_connues_credibles = [
    "digitalspy.com",
    "tennessean.com",
    "seventeen.com"
]

sources_peu_connues_peu_credibles = [
    "nationalenquirer.com",
    "inquisitr.com",
    "celebritynetworth.com"
]


MODS = ["gemma3:4b", "qwen2.5:7b", "llama3.2:3b", "phi4-mini"]

MODELES = ["gemma3:4b", "gemma3:1b", "gemma3:12b",
           "qwen2.5:1.5b", "qwen2.5:7b", "qwen2.5:14b",
           "llama3.2:1b", "llama3.2:3b", "phi4-mini"]

preuve = True  # True seulement pour le premier modèle avec RAG

for m in MODS:

    nom = m.replace(":", "_")

    print("==================== AVEC RAG :", m, "=================================")

    lancer_test_contrefactuel(echantillon=articles, 
                              chemin_sortie=chemin_out + nom + "_RAG.csv",
                              sources_connues_credibles=sources_connues_credibles,
                              sources_connues_peu_credibles=sources_connues_peu_credibles,
                              sources_peu_connues_credibles=sources_peu_connues_credibles,
                              sources_peu_connues_peu_credibles=sources_peu_connues_peu_credibles,
                              avec_rag=True,
                              creer_preuves=preuve)

    preuve = False

    print("==================== SANS RAG :", m, "=================================")

    lancer_test_contrefactuel(echantillon=articles, 
                              chemin_sortie=chemin_out + nom + "_WRAG.csv",
                              sources_connues_credibles=sources_connues_credibles,
                              sources_connues_peu_credibles=sources_connues_peu_credibles,
                              sources_peu_connues_credibles=sources_peu_connues_credibles,
                              sources_peu_connues_peu_credibles=sources_peu_connues_peu_credibles,
                              avec_rag=False,
                              creer_preuves=False)

==================== AVEC RAG : gemma3:4b =================================
RAG CRÉÉ pour : gossipcop-6428453817
[1/150] OK
RAG CRÉÉ pour : gossipcop-1812612510
[2/150] OK
RAG CRÉÉ pour : politifact14896
[3/150] OK
RAG CRÉÉ pour : politifact8085
[4/150] OK
RAG CRÉÉ pour : politifact8371
[5/150] OK
RAG CRÉÉ pour : politifact6646
[6/150] OK
RAG CRÉÉ pour : politifact627
[7/150] OK
RAG CRÉÉ pour : politifact11899
[8/150] OK
  [!] Tentative 1/3 echouee : No results found.
RAG CRÉÉ pour : politifact15423
[9/150] OK
  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 echouee : No results found.
  [!] Tentative 3/3 echouee : No results found.
RAG CRÉÉ pour : politifact15379
[10/150] OK
RAG CRÉÉ pour : politifact9107
[11/150] OK
RAG CRÉÉ pour : politifact15309
[12/150] OK
  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 echouee : No results found.
RAG CRÉÉ pour : politifact11709
[13/150] OK
  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 e

h2 connection driver error: connection reset


  [!] Tentative 1/3 echouee : DecodeError: DecodeError('error decoding response body > request or response body error > error sending request > connection reset', None)
RAG CRÉÉ pour : gossipcop-2723098494
[67/150] OK
RAG CRÉÉ pour : politifact14944
[68/150] OK
RAG CRÉÉ pour : gossipcop-876260
[69/150] OK
  [!] Tentative 1/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
RAG CRÉÉ pour : politifact14516
[70/150] OK
RAG CRÉÉ pour : politifact379
[71/150] OK
  [!] Tentative 1/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
RAG CRÉÉ pour : gossipcop-900207
[72/150] OK
RAG CRÉÉ pour : gossipcop-5310842420
[73/150] OK
RAG CRÉÉ pour : politifact14855
[74/150] OK
RAG CRÉÉ pour : gossipcop-862435
[75/150] OK
  [!] Tentative 1/3 echouee : No results found.
  [!] Tentative 2/3 echouee : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
  [!] Tentative 3/3 ec

**TEST COCHRAN**


In [ ]:
def calcul_cochran_q(chemin_csv, nom, col_connu, col_peu_connu, col_anonyme):
    articles = lire_csv(chemin_csv)

    tab=[]
    for a in articles:
        if a[col_connu] == "real":
            connu = 1
        else:
            connu = 0

        if a[col_peu_connu] == "real":
            peu_connu = 1
        else:
            peu_connu = 0

        if a[col_anonyme] == "real":
            anonyme = 1
        else:
            anonyme = 0

        tab.append([connu, peu_connu, anonyme])
        
    resultat = cochrans_q(tab)

    print(f"=== {nom} ===")
    print(f"  Statistique Q : {resultat.statistic:.2f}")
    print(f"  p-value : {resultat.pvalue:.2e}")

    if resultat.pvalue < 0.001:
        print("  => Les 3 conditions different significativement entre elles (p<0.001)")
    else:
        print("  => Pas de difference significative globale")
    print()

    return resultat
    

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/"

resultat_B_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_B_RAG.csv",
    nom="Categorie B (avec RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_D_RAG.csv",
    nom="Categorie D (avec RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

resultat_B_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_B_WRAG.csv",
    nom="Categorie B (sans RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "qwen_D_WRAG.csv",
    nom="Categorie D (sans RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

**Test mcneamar**

In [ ]:
 
def comparer_deux_colonnes(chemin_csv, nom_colonne_1, nom_colonne_2):
    # cette fonction compare 2 colonnes dun fichier csv avec le test mcNeamar
    # la fonction retourne un dictionnaire avec b, c, p-value,
    # proportions "real" de chaque colonne, diff + IC95%
    # b: nb d'articles "real" (colonne 1) devenus "fake" (colonne 2)
    # c: nb d'articles "fake" (colonne 1) devenus "real" (colonne 2)
    # p_value: probabilite d'observer ce desequilibre entre b et c
    # p_real_colonne_1: proportion d'articles "real" dans la colonne 1
    # p_real_colonne_2: proportion d'articles "real" dans la colonne 2
    # diff_proportions: variation de la proportion de "real" (colonne_2 - colonne_1)
    # ic95_low/high:(95% de confiance) la plus petite et la plus grande valeur probable de la "vraie" difference
 
    col1, col2 = [], []
    with open(chemin_csv, newline="", encoding="utf-8") as f:
        for ligne in csv.DictReader(f):
            col1.append(ligne[nom_colonne_1].strip().lower())
            col2.append(ligne[nom_colonne_2].strip().lower())
 
    n = len(col1)
 
    table = {"fake_fake": 0, "fake_real": 0, "real_fake": 0, "real_real": 0}
    for a, b_ in zip(col1, col2):
        table[f"{a}_{b_}"] += 1
 
    b = table["real_fake"]   # col1=real, col2=fake 
    c = table["fake_real"]   # col1=fake, col2=real
    nbr_flip = b + c
 
    # mcnemar
    if nbr_flip == 0:
        stat, p = 0.0, 1.0
    else:
        stat = (abs(b - c) - 1) ** 2 / nbr_flip
        p = 1 - chi2.cdf(stat, df=1)
 
    # proportions + IC95% de la difference appariee
    p1 = col1.count("real") / n #proportion d'articles "real" dans la colonne 1
    p2 = col2.count("real") / n #proportion d'articles "real" dans la colonne 2
    
    diff = p2 - p1
    
    se = ((b + c) - (b - c) ** 2 / n) ** 0.5 / n
    ic_low, ic_high = diff - 1.96 * se, diff + 1.96 * se
 
    return {
        "colonne_1": nom_colonne_1,
        "colonne_2": nom_colonne_2,
        "b_real_to_fake": b,
        "c_fake_to_real": c,
        "p_value": p,
        "p_real_colonne_1": p1,
        "p_real_colonne_2": p2,
        "diff_proportions": diff,
        "ic95_low": ic_low,
        "ic95_high": ic_high,
    }

In [ ]:
 
def afficher_resultat(res, alpha=0.05/3):
    #Afficher les resultats de comparer_deux_colonnes() de facon lisible
    
    print(f"Comparaison : {res['colonne_1']} vs {res['colonne_2']}")
    print(f"  b (real->fake) = {res['b_real_to_fake']}, c (fake->real) = {res['c_fake_to_real']}, "
          f"changements = {res['b_real_to_fake'] + res['c_fake_to_real']}")
    print(f"  McNemar : p = {res['p_value']:.4g} "
          f"-> {'significatif' if res['p_value'] < alpha else 'non significatif'} (alpha={alpha})")
    print(f"  Proportion 'real' : {res['p_real_colonne_1']*100:.1f}% -> {res['p_real_colonne_2']*100:.1f}%")
    print(f"  Difference = {res['diff_proportions']*100:+.1f} points, "
          f"IC95% = [{res['ic95_low']*100:+.1f} ; {res['ic95_high']*100:+.1f}]")
    print()
 

 

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_B_RAG.csv"
print("Gemma sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_B_WRAG.csv"
print("Gemma sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_D_RAG.csv"
print("Gemma sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/gemma_D_WRAG.csv"
print("Gemma sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_B_RAG.csv"
print("Qwen sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_B_WRAG.csv"
print("Qwen sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_D_RAG.csv"
print("Qwen sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/experiences-results/qwen_D_WRAG.csv"
print("Qwen sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)